In [5]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().parent  # ou Path.cwd()
sys.path.append(str(PROJECT_ROOT))
from src.models.architecture import DeepfakeClassifier
from src.data.dataset import get_dataloaders


In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = DeepfakeClassifier(
    model_name='efficientnet_b0',
    num_classes=2,
    pretrained=False
)


Model: efficientnet_b0
Feature dim: 1280
Num classes: 2
Pretrained: False


In [11]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent   # adapte si besoin
MODEL_PATH = PROJECT_ROOT / "models" / "best_model.pth"

print(MODEL_PATH)
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint)
model = model.to(device)
model.eval()

print("✅ Modèle chargé!")

# Cellule 3: Charger le test set
_, _, test_loader = get_dataloaders(
    "data/processed",
    batch_size=32
)

c:\Users\PI\Documents\deepguard-detection\models\best_model.pth
✅ Modèle chargé!
Dataset train: 0 images
  - Real: 0
  - Fake: 0
Dataset val: 0 images
  - Real: 0
  - Fake: 0
Dataset test: 0 images
  - Real: 0
  - Fake: 0


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [8]:
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

NameError: name 'test_loader' is not defined

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
print(classification_report(
    all_labels, all_preds,
    target_names=['Real', 'Fake']
))

In [ ]:
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
indices = np.random.choice(len(all_labels), 8, replace=False)

for i, idx in enumerate(indices):
    # Récupérer l'image
    image, label = test_loader.dataset[idx]
    
    # Dénormaliser
    image = image.permute(1, 2, 0).numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = image * std + mean
    image = np.clip(image, 0, 1)
    
    # Afficher
    ax = axes[i//4, i%4]
    ax.imshow(image)
    
    true_label = 'REAL' if label == 0 else 'FAKE'
    pred_label = 'REAL' if all_preds[idx] == 0 else 'FAKE'
    prob = all_probs[idx] if all_preds[idx] == 1 else (1 - all_probs[idx])
    
    color = 'green' if all_preds[idx] == label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({prob:.2%})',
                 color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()